Week 10

This notebook uses the Universal Sentence Encoder (USE) to generate embeddings and a Deep Neural Network (DNN) for classification.

I first uploaded the necessary libraries as well as the actual data through a url becuase my version of jupyter on the cloud blocks large uploads.

In [1]:
import pandas as pd
import numpy as np



url = "https://raw.githubusercontent.com/bellevue-university/dsc360/main/12%20Week/week_10/data/hotel-reviews.csv"
df = pd.read_csv(url)
print(df.columns)

df = pd.read_csv(url, usecols=['Description', 'Is_Response'])

print(df.head())
print(df['Is_Response'].value_counts())

Index(['User_ID', 'Description', 'Browser_Used', 'Device_Used', 'Is_Response'], dtype='object')
                                         Description Is_Response
0  The room was kind of clean but had a VERY stro...   not happy
1  I stayed at the Crown Plaza April -- - April -...   not happy
2  I booked this hotel through Hotwire at the low...   not happy
3  Stayed here with husband and sons on the way t...       happy
4  My girlfriends and I stayed here to celebrate ...   not happy
Is_Response
happy        26521
not happy    12411
Name: count, dtype: int64


The data is extracted from the two necessary columns (description and the label). Some cleaning is done.

In [2]:
import re
from sklearn.model_selection import train_test_split

# 1. Clean the text (Remove punctuation and lowercase)
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['clean_review'] = df['Description'].apply(clean_text)



The labels are mapped onto numbers, happy = 1 and not happy = 0

In [3]:

df['sentiment_label'] = df['Is_Response'].map({'happy': 1, 'not happy': 0})



I sampled the data, only using 5000 random reviews from the set so that it runs smoothly on the cloud enviornment.

In [4]:

df_sample = df.sample(n=5000, random_state=42)



The data is split into test and train %80 of the data is to train the model, and %20 is to test how the model is working.

In [5]:
# 4. Train/Test Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_sample['clean_review'], 
    df_sample['sentiment_label'], 
    test_size=0.2, 
    random_state=42
)

print("Preprocessing complete.")
print(f"Training samples: {len(X_train_raw)}")
print(f"Testing samples: {len(X_test_raw)}")

Preprocessing complete.
Training samples: 4000
Testing samples: 1000


The following cell contains code to install the tensorflow package. I didn't have enough space on the disk so it took me some time to clear space and try different methods until it worked.

In [6]:
#I put this code in after the next cell when i realized i didn't have this package installed.
#i have no space pn my disk so i will attempt to load it into a temperaery system.
#!pip cache purge

!pip install --user tensorflow-hub


Looking in links: /usr/share/pip-wheels


In [7]:
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.metrics import classification_report, accuracy_score



2026-02-27 18:22:27.547595: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-27 18:22:27.623748: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-27 18:22:30.044093: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/f8bded8e-df7e-4491-93d9-90b7721edcf0/.local/lib/python3.12/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import pars

Loading The UNIVERSAL SENTENCE ENCODER (The "Chapter 10" Model)

In [8]:
module_url = "https://tfhub.dev/google/universal-sentence-encoder/4"
model_use = hub.load(module_url)



2026-02-27 18:22:31.712286: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


ENCODING THE TEXT DATA
the raw text is converted into numeric vectors that the neural network can understand

In [9]:

import numpy as np
print("Encoding text... this may take a moment.")
X_train_encoded = np.array(model_use(X_train_raw.tolist()))
X_test_encoded = np.array(model_use(X_test_raw.tolist()))



Encoding text... this may take a moment.


2026-02-27 18:22:41.343422: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 1587755520 exceeds 10% of free system memory.


The Nueral networks architecture is defined. 

In [10]:

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(512,)), # Input is the 512D vector from USE
    tf.keras.layers.Dense(128, activation='relu'), # Hidden layer
    tf.keras.layers.Dropout(0.2),                 # Prevents overfitting
    tf.keras.layers.Dense(64, activation='relu'),  # Hidden layer
    tf.keras.layers.Dense(1, activation='sigmoid') # Output layer (0 or 1)
])



the model is compiled - the model uses adam, binary crossentropy and is mesured with accuracy. The most important number for data scientists is the overall accuracy.

In [11]:

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])



The model is trained one part at a time. as the model trains more the loss lowers and the accuracy rises from the first to the last round the accuracy rose over %10. 

In [12]:

print("\nStarting Training...")
history = model.fit(
    X_train_encoded, y_train, 
    epochs=10, 
    batch_size=32, 
    validation_split=0.1, 
    verbose=1
)




Starting Training...
Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7850 - loss: 0.4512 - val_accuracy: 0.8600 - val_loss: 0.3531
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8614 - loss: 0.3294 - val_accuracy: 0.8575 - val_loss: 0.3631
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8692 - loss: 0.3122 - val_accuracy: 0.8500 - val_loss: 0.3450
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8717 - loss: 0.2977 - val_accuracy: 0.8425 - val_loss: 0.3436
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8772 - loss: 0.2853 - val_accuracy: 0.8500 - val_loss: 0.3425
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8858 - loss: 0.2766 - val_accuracy: 0.8425 - val_loss: 0.3692
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8889 - loss: 0.2657 - val_accuracy: 0.8400 - val_loss: 0.3605
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8922 - loss: 0.2

In [14]:

predictions = (model.predict(X_test_encoded) > 0.5).astype("int32")

print("\n=== MODEL EVALUATION METRICS ===")
print(f"Final Accuracy: {accuracy_score(y_test, predictions):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=['not happy', 'happy']))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

=== MODEL EVALUATION METRICS ===
Final Accuracy: 0.8470

Classification Report:
              precision    recall  f1-score   support

   not happy       0.77      0.75      0.76       318
       happy       0.88      0.89      0.89       682

    accuracy                           0.85      1000
   macro avg       0.82      0.82      0.82      1000
weighted avg       0.85      0.85      0.85      1000



The overall accuracy is %85 percent which is very good. The f1 score is also %85 but it is split over .90 for the happy , which means %90 of the happy review are detected. this is great. BUt the not happy reviews are only detected %76 percent of the time. The result of the model is very possibly due to the fact that the training and testing data had a lot more happy reviews in it.